LOAD AND PROCESS DATA

In [ ]:
import os
import numpy as np

import tensorflow as tf
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv2D, MaxPooling2D, Flatten, Dense, Dropout, BatchNormalization, GlobalAveragePooling2D
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau

from sklearn.metrics import classification_report, confusion_matrix
import seaborn as sns
import matplotlib.pyplot as plt

In [ ]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("bhaveshmittal/melanoma-cancer-dataset")

print("Path to dataset files:", path)

In [ ]:
BASE_DIR = path  # Change based on your dataset location
TRAIN_DIR = os.path.join(BASE_DIR, "train")
TEST_DIR = os.path.join(BASE_DIR, "test")

# Image size and batch size
IMG_SIZE = (224, 224)  # Optimal size for EfficientNetB0
BATCH_SIZE = 32

# Data Augmentation for Training
train_datagen = ImageDataGenerator(
    rescale=1./255,  # Normalize pixel values to [0, 1]
    rotation_range=20,
    width_shift_range=0.2,
    height_shift_range=0.2,
    shear_range=0.2,
    zoom_range=0.2,
    horizontal_flip=True
)

# Rescaling only for Testing
test_datagen = ImageDataGenerator(rescale=1./255)

# Load Training Data
train_generator = train_datagen.flow_from_directory(
    directory=TRAIN_DIR,
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode="binary"  # Binary classification: Benign vs Malignant
)

# Load Testing Data
test_generator = test_datagen.flow_from_directory(
    directory=TEST_DIR,
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode="binary"
)

# Print class labels
print("Class Mapping:", train_generator.class_indices)

In [ ]:
import cv2
import time
import joblib

from skimage.feature import hog

from sklearn.preprocessing import StandardScaler

from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score
)

In [ ]:
# HOG Feature Extraction

ML_SIZE = 64

def extract_hog_features(folder):

    X = []
    y = []

    classes = ["Benign", "Malignant"]

    for label, class_name in enumerate(classes):

        class_path = os.path.join(folder, class_name)

        for img_name in os.listdir(class_path):

            img_path = os.path.join(class_path, img_name)

            img = cv2.imread(img_path)

            if img is None:
                continue

            img = cv2.resize(img, (ML_SIZE, ML_SIZE))

            gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)

            features = hog(
                gray,
                orientations=9,
                pixels_per_cell=(8,8),
                cells_per_block=(2,2)
            )

            X.append(features)
            y.append(label)

    return np.array(X), np.array(y)

In [ ]:
# Load ML Dataset

X_train_ml, y_train_ml = extract_hog_features(TRAIN_DIR)

X_test_ml, y_test_ml = extract_hog_features(TEST_DIR)

print(X_train_ml.shape)
print(X_test_ml.shape)

In [ ]:
# Scaling

scaler = StandardScaler()

X_train_ml = scaler.fit_transform(X_train_ml)

X_test_ml = scaler.transform(X_test_ml)

LOGISTIC REGRESSION

In [ ]:
start = time.time()

lr = LogisticRegression(max_iter=1000)

lr.fit(X_train_ml, y_train_ml)

lr_pred = lr.predict(X_test_ml)

lr_time = time.time() - start

KNN

In [ ]:
start = time.time()

knn = KNeighborsClassifier(n_neighbors=5)

knn.fit(X_train_ml, y_train_ml)

knn_pred = knn.predict(X_test_ml)

knn_time = time.time() - start

RANDOM FOREST

In [ ]:
start = time.time()

rf = RandomForestClassifier(
    n_estimators=200,
    random_state=42
)

rf.fit(X_train_ml, y_train_ml)

rf_pred = rf.predict(X_test_ml)

rf_time = time.time() - start

SVM

In [ ]:
start = time.time()

svm = SVC(
    kernel="rbf",
    C=10
)

svm.fit(X_train_ml, y_train_ml)

svm_pred = svm.predict(X_test_ml)

svm_time = time.time() - start

MATRICS FUNCTION

In [ ]:
def evaluate_model(name, y_true, y_pred):

    acc = accuracy_score(y_true, y_pred)

    prec = precision_score(y_true, y_pred)

    rec = recall_score(y_true, y_pred)

    f1 = f1_score(y_true, y_pred)

    print("\n", name)

    print("Accuracy :", acc)

    print("Precision:", prec)

    print("Recall   :", rec)

    print("F1 Score :", f1)

    print(classification_report(y_true, y_pred))

    return acc, prec, rec, f1

In [ ]:
lr_metrics = evaluate_model(
    "Logistic Regression",
    y_test_ml,
    lr_pred
)

knn_metrics = evaluate_model(
    "KNN",
    y_test_ml,
    knn_pred
)

rf_metrics = evaluate_model(
    "Random Forest",
    y_test_ml,
    rf_pred
)

svm_metrics = evaluate_model(
    "SVM",
    y_test_ml,
    svm_pred
)

In [ ]:
# Display some sample images from dataset


def plot_images(generator):

    images, labels = next(generator)

    plt.figure(figsize=(10,10))

    for i in range(min(9, len(images))):

        plt.subplot(3,3,i+1)

        plt.imshow(images[i])

        if labels[i] == 1:
            title = "Malignant (Cancer)"
        else:
            title = "Benign (Non-Cancer)"

        plt.title(title)

        plt.axis("off")

    plt.tight_layout()

    plt.show()


plot_images(train_generator)

BUILD CNN MODEL

In [ ]:
from tensorflow.keras import layers, models

# Build the CNN model
model = models.Sequential([
    # 1st Convolution Block
    layers.Conv2D(32, (3, 3), activation='relu', input_shape=(224, 224, 3)),
    layers.MaxPooling2D((2, 2)),

    # 2nd Convolution Block
    layers.Conv2D(64, (3, 3), activation='relu'),
    layers.MaxPooling2D((2, 2)),

    # 3rd Convolution Block
    layers.Conv2D(128, (3, 3), activation='relu'),
    layers.MaxPooling2D((2, 2)),

    # Flattening Layer
    layers.Flatten(),

    # Fully Connected Layers
    layers.Dense(512, activation='relu'),
    layers.Dropout(0.5),  # Dropout to reduce overfitting
    layers.Dense(1, activation='sigmoid')  # Output layer for binary classification
])

# Compile the model
model.compile(optimizer='adam',  # Optimizer for faster convergence
              loss='binary_crossentropy',  # Binary classification loss
              metrics=['accuracy'])  # Accuracy metric

# Model Summary
model.summary()

TRAIN AND EVALIATE MODEL


In [ ]:
# EarlyStopping
early_stopping = EarlyStopping(
    monitor="val_loss",
    patience=2,              # faster stopping
    restore_best_weights=True,
    verbose=1
)

# Reduce learning rate faster
lr_scheduler = ReduceLROnPlateau(
    monitor="val_loss",
    factor=0.5,
    patience=1,
    min_lr=1e-6,
    verbose=1
)

history = model.fit(
    train_generator,
    steps_per_epoch=100,     # reduced steps → faster training
    epochs=5,               # reduced epochs
    validation_data=test_generator,
    validation_steps=20,     # reduced validation steps
    callbacks=[early_stopping, lr_scheduler]
)

In [ ]:
# Save Model
model.save("skin_cancer_cnn.h5")

# Plot Training History
plt.plot(history.history["accuracy"], label="Train Accuracy")
plt.plot(history.history["val_accuracy"], label="Validation Accuracy")
plt.xlabel("Epochs")
plt.ylabel("Accuracy")
plt.legend()
plt.show()

In [ ]:
import pandas as pd

# Predict on the test set
test_pred = model.predict(test_generator, steps=test_generator.samples // BATCH_SIZE, verbose=1)

# Convert predictions to binary labels (0 or 1)
test_pred_labels = (test_pred > 0.5).astype("int32")

# Get the true labels
test_true_labels = test_generator.classes[:len(test_pred_labels)]  # Match length to predictions

# ==================================================
# CNN METRICS
# ==================================================

cnn_acc = accuracy_score(
    test_true_labels,
    test_pred_labels
)

cnn_prec = precision_score(
    test_true_labels,
    test_pred_labels
)

cnn_rec = recall_score(
    test_true_labels,
    test_pred_labels
)

cnn_f1 = f1_score(
    test_true_labels,
    test_pred_labels
)

print("\nCNN Performance Metrics")
print("Accuracy :", cnn_acc)
print("Precision:", cnn_prec)
print("Recall   :", cnn_rec)
print("F1 Score :", cnn_f1)


# ==================================================
# CONFUSION MATRIX FUNCTION
# ==================================================

def plot_cm(y_true, y_pred, title):

    cm = confusion_matrix(
        y_true,
        y_pred
    )

    plt.figure(figsize=(6,5))

    sns.heatmap(
        cm,
        annot=True,
        fmt='d',
        cmap='Blues',
        xticklabels=['Benign','Malignant'],
        yticklabels=['Benign','Malignant']
    )

    plt.title(title)

    plt.xlabel("Predicted")

    plt.ylabel("Actual")

    plt.show()


# ==================================================
# CONFUSION MATRICES FOR ALL MODELS
# ==================================================

plot_cm(
    y_test_ml,
    lr_pred,
    "Logistic Regression"
)

plot_cm(
    y_test_ml,
    knn_pred,
    "KNN"
)

plot_cm(
    y_test_ml,
    rf_pred,
    "Random Forest"
)

plot_cm(
    y_test_ml,
    svm_pred,
    "SVM"
)

plot_cm(
    test_true_labels,
    test_pred_labels,
    "CNN"
)


# ==================================================
# FINAL COMPARISON TABLE
# ==================================================

results = pd.DataFrame({

    "Model":[
        "Logistic Regression",
        "KNN",
        "Random Forest",
        "SVM",
        "CNN"
    ],

    "Accuracy":[
        lr_metrics[0],
        knn_metrics[0],
        rf_metrics[0],
        svm_metrics[0],
        cnn_acc
    ],

    "Precision":[
        lr_metrics[1],
        knn_metrics[1],
        rf_metrics[1],
        svm_metrics[1],
        cnn_prec
    ],

    "Recall":[
        lr_metrics[2],
        knn_metrics[2],
        rf_metrics[2],
        svm_metrics[2],
        cnn_rec
    ],

    "F1 Score":[
        lr_metrics[3],
        knn_metrics[3],
        rf_metrics[3],
        svm_metrics[3],
        cnn_f1
    ]

})

print("\nModel Comparison")
display(results)


# ==================================================
# ACCURACY COMPARISON GRAPH
# ==================================================

plt.figure(figsize=(10,5))

sns.barplot(
    data=results,
    x="Model",
    y="Accuracy"
)

plt.title("Accuracy Comparison")

plt.xticks(rotation=15)

plt.show()


# ==================================================
# PRECISION COMPARISON GRAPH
# ==================================================

plt.figure(figsize=(10,5))

sns.barplot(
    data=results,
    x="Model",
    y="Precision"
)

plt.title("Precision Comparison")

plt.xticks(rotation=15)

plt.show()


# ==================================================
# RECALL COMPARISON GRAPH
# ==================================================

plt.figure(figsize=(10,5))

sns.barplot(
    data=results,
    x="Model",
    y="Recall"
)

plt.title("Recall Comparison")

plt.xticks(rotation=15)

plt.show()


# ==================================================
# F1 SCORE COMPARISON GRAPH
# ==================================================

plt.figure(figsize=(10,5))

sns.barplot(
    data=results,
    x="Model",
    y="F1 Score"
)

plt.title("F1 Score Comparison")

plt.xticks(rotation=15)

plt.show()


# ==================================================
# BEST MODEL
# ==================================================

best_model = results.sort_values(
    by="Accuracy",
    ascending=False
)

print("\nBest Model Based On Accuracy")
display(best_model)

In [ ]:
plt.figure(figsize=(10,5))

sns.barplot(
    data=results,
    x="Model",
    y="Accuracy"
)

plt.title(
    "Model Accuracy Comparison"
)

plt.xticks(rotation=15)

plt.show()

EVALUATE THE PERFORMANCE

In [ ]:
# Predict on the test set
test_pred = model.predict(test_generator, steps=test_generator.samples // BATCH_SIZE, verbose=1)

# Convert predictions to binary labels (0 or 1)
test_pred_labels = (test_pred > 0.5).astype("int32")

# Get the true labels
test_true_labels = test_generator.classes[:len(test_pred_labels)]  # Match length to predictions

# Classification report
print("Classification Report:")
print(classification_report(test_true_labels, test_pred_labels))

# Confusion Matrix
cm = confusion_matrix(test_true_labels, test_pred_labels)

# Plot Confusion Matrix
plt.figure(figsize=(6, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=train_generator.class_indices.keys(), yticklabels=train_generator.class_indices.keys())
plt.xlabel('Predicted')
plt.ylabel('True')
plt.title('Confusion Matrix')
plt.show()

PREDICTION SYSTEM

In [ ]:
from tensorflow.keras.preprocessing import image
from tensorflow.keras.models import load_model

# Load the entire model
model = load_model('/content/skin_cancer_cnn.h5')


def predict_skin_cancer(image_path, model):
    img = image.load_img(image_path, target_size=(224, 224))  # Load Image
    img_array = image.img_to_array(img) / 255.0  # Normalize
    img_array = np.expand_dims(img_array, axis=0)  # Add batch dimension

    # Make Prediction
    prediction = model.predict(img_array)
    class_label = "Malignant" if prediction > 0.5 else "Benign"

    # Show Image with Prediction
    plt.imshow(img)
    plt.title(f"Predicted: {class_label}")
    plt.axis("off")
    plt.show()

In [ ]:
# =====================================================
# FINAL PROJECT DEMONSTRATION CELL
# =====================================================

import numpy as np
import matplotlib.pyplot as plt
import cv2

from google.colab import files
from tensorflow.keras.preprocessing import image

print("="*60)
print("SKIN CANCER DETECTION PROJECT")
print("="*60)


# CHECK CNN MODEL


if 'cnn_model' in globals():

    prediction_model = cnn_model

elif 'model' in globals():

    prediction_model = model

else:

    raise Exception(
        "CNN model not found. Train the CNN first or load it using:\n"
        "from tensorflow.keras.models import load_model\n"
        "model = load_model('skin_cancer_cnn.h5')"
    )

# =====================================================
# MODEL COMPARISON
# =====================================================

print("\nMODEL PERFORMANCE COMPARISON\n")

display(results)

best_model = results.loc[
    results["Accuracy"].idxmax()
]

print("\nBEST MODEL")

print("Model      :", best_model["Model"])

print("Accuracy   :",
      round(best_model["Accuracy"]*100,2),
      "%")

print("Precision  :",
      round(best_model["Precision"]*100,2),
      "%")

print("Recall     :",
      round(best_model["Recall"]*100,2),
      "%")

print("F1 Score   :",
      round(best_model["F1 Score"]*100,2),
      "%")

# =====================================================
# UPLOAD IMAGE
# =====================================================

print("\nUpload Skin Image")

uploaded = files.upload()

img_path = list(uploaded.keys())[0]

# =====================================================
# DISPLAY IMAGE
# =====================================================

plt.figure(figsize=(6,6))

img_display = plt.imread(img_path)

plt.imshow(img_display)

plt.axis("off")

plt.title("Uploaded Image")

plt.show()

# =====================================================
# CNN PREDICTION
# =====================================================

img = image.load_img(
    img_path,
    target_size=(224,224)
)

img_array = image.img_to_array(img)

img_array = img_array.astype("float32") / 255.0

img_array = np.expand_dims(
    img_array,
    axis=0
)

cnn_prob = prediction_model.predict(
    img_array,
    verbose=0
)[0][0]

if cnn_prob > 0.5:

    cnn_result = "Malignant (Cancer)"

else:

    cnn_result = "Benign (Non-Cancer)"

cnn_conf = round(
    max(cnn_prob, 1-cnn_prob) * 100,
    2
)

# =====================================================
# SVM PREDICTION
# =====================================================

img_ml = cv2.imread(img_path)

img_ml = cv2.resize(
    img_ml,
    (64,64)
)

gray = cv2.cvtColor(
    img_ml,
    cv2.COLOR_BGR2GRAY
)

hog_features = hog(
    gray,
    orientations=9,
    pixels_per_cell=(8,8),
    cells_per_block=(2,2)
)

hog_features = hog_features.reshape(
    1,
    -1
)

hog_features = scaler.transform(
    hog_features
)

svm_prediction = svm.predict(
    hog_features
)[0]

if svm_prediction == 1:

    svm_result = "Malignant (Cancer)"

else:

    svm_result = "Benign (Non-Cancer)"

# =====================================================
# RESULTS
# =====================================================

print("\n" + "="*60)

print("PREDICTION RESULTS")

print("="*60)

print("\nCNN RESULT")

print("Prediction :", cnn_result)

print("Confidence :", cnn_conf, "%")

print("\nSVM RESULT")

print("Prediction :", svm_result)

# =====================================================
# FINAL DECISION
# =====================================================

print("\n" + "="*60)

print("FINAL DIAGNOSIS")

print("="*60)

if cnn_result == svm_result:

    final_result = cnn_result

else:

    final_result = cnn_result

print("\nPrediction :", final_result)

if "Malignant" in final_result:

    print("\nWARNING: Possible skin cancer detected.")

else:

    print("\nNo cancer indication detected.")

print("\nProject Completed Successfully.")